In [5]:
import pandas as pd
import requests
from io import StringIO

base_url = "https://en.wikipedia.org/wiki/List_of_largest_technology_companies_by_revenue"
respuesta = requests.get(base_url, headers={"User-Agent": "proyecto-bootcamp"})
print(respuesta.status_code)

tablas = pd.read_html(StringIO(respuesta.text))
len(tablas)

200


30

In [6]:
for i, t in enumerate(tablas):
    print(i, t.shape)

0 (29, 5)
1 (24, 5)
2 (21, 5)
3 (20, 5)
4 (20, 5)
5 (17, 8)
6 (18, 7)
7 (17, 7)
8 (18, 8)
9 (14, 2)
10 (3, 2)
11 (3, 2)
12 (4, 2)
13 (58, 2)
14 (5, 2)
15 (1, 2)
16 (2, 2)
17 (2, 2)
18 (5, 2)
19 (2, 2)
20 (2, 2)
21 (3, 2)
22 (2, 2)
23 (3, 2)
24 (2, 2)
25 (8, 2)
26 (6, 2)
27 (3, 2)
28 (2, 2)
29 (2, 2)


In [7]:
tablas[0].head()

,Company,Revenue ($B) USD[1],Profit ($B) USD[1],Country (origin),Headquarters
0,Apple,$416.2,$112.0,US,"Cupertino, California, US"
1,Alphabet,$402.8,$132.2,US,"Mountain View, California, US"
2,Microsoft,$281.7,$101.8,US,"Redmond, Washington, US"
3,Foxconn,$260.1,$6.1,Taiwan,"New Taipei City, Taiwan"
4,Samsung Electronics,$234.7,$31.1,South Korea,"Suwon, South Korea"


In [8]:
empresas = ["Apple", "Amazon", "Microsoft", "Alphabet", "Meta"]
df = tablas[0]
df_filtrado = df[df["Company"].str.contains("|".join(empresas), case=False)]
df_filtrado

,Company,Revenue ($B) USD[1],Profit ($B) USD[1],Country (origin),Headquarters
0,Apple,$416.2,$112.0,US,"Cupertino, California, US"
1,Alphabet,$402.8,$132.2,US,"Mountain View, California, US"
2,Microsoft,$281.7,$101.8,US,"Redmond, Washington, US"
6,Meta,$201.0,$60.5,US,"Menlo Park, California, US"


In [9]:
for i, t in enumerate(tablas):
    if t.astype(str).apply(lambda col: col.str.contains("Amazon")).any().any():
        print(i)

8
13
18
20
21
27


In [10]:
display(tablas[8].head())
display(tablas[13].head())

,0,1,2,3,4,5,6,7
0,NaN,Rank[25],Company,Fiscal year ending,Revenue ($B) USD,Employees,Revenue per employee ($K USD),Headquarters
1,1.0,NaN,Apple Inc.,"September 30, 2017[26]",$229.2[25][27],"123,000[27]",$1863.41463,"Cupertino, California, US"
2,2.0,NaN,Samsung Electronics,"December 31, 2017[28]",$211.9[25][29][30],"320,670[31][32]",$660.80394,"Suwon, South Korea"
3,3.0,NaN,Amazon,"December 31, 2017[33][34]",$177.9[25][34],"613,300[35]",$290.07011,"Seattle, Washington, US"
4,4.0,NaN,Foxconn,"December 31, 2017[36][37]",$154.7–158[25][37][38],"803,126[39]",$192.62232,"New Taipei City, Taiwan"


,Major technology companies,Major technology companies.1
0,"vteMajor personal computer, server, and mainfr...","vteMajor personal computer, server, and mainfr..."
1,"vteMajor personal computer, server, and mainfr...","vteMajor personal computer, server, and mainfr..."
2,Companies with annual revenue of over US$3 bil...,Companies with annual revenue of over US$3 bil...
3,Personal computers and servers,Acer Apple Asus Dell Fujitsu Huawei HP Lenovo ...
4,Servers only,Cisco EMC HPE IBM Inspur NetApp Oracle


Fuente: Wikipedia, “List of largest technology companies by revenue”. Probado con requests + pd.read_html: la página devuelve 200 y se leen 30 tablas. La tabla principal (la más reciente) trae ingresos y beneficios en miles de millones de USD de Apple, Alphabet, Microsoft y Meta. Amazon aparece solo en tablas de años anteriores (por ejemplo 2017). Los datos traen notas de Wikipedia ([25]) y $ que hay que limpiar. Pendiente: limpiar y guardar en data/raw.

In [11]:
df = tablas[0].copy()


In [12]:
df.columns = ["empresa", "ingresos_mmusd", "beneficio_mmusd", "pais", "sede"]

In [13]:
df["ingresos_mmusd"] = df["ingresos_mmusd"].str.replace("$", "", regex=False)

In [14]:
df.head()

,empresa,ingresos_mmusd,beneficio_mmusd,pais,sede
0,Apple,416.2,$112.0,US,"Cupertino, California, US"
1,Alphabet,402.8,$132.2,US,"Mountain View, California, US"
2,Microsoft,281.7,$101.8,US,"Redmond, Washington, US"
3,Foxconn,260.1,$6.1,Taiwan,"New Taipei City, Taiwan"
4,Samsung Electronics,234.7,$31.1,South Korea,"Suwon, South Korea"


In [17]:
df["beneficio_mmusd"] = (
    df["beneficio_mmusd"]
    .str.replace("(", "-", regex=False)
    .str.replace(")", "", regex=False)
)
df["beneficio_mmusd"] = pd.to_numeric(df["beneficio_mmusd"])
df.dtypes

empresa                str
ingresos_mmusd     float64
beneficio_mmusd    float64
pais                   str
sede                   str
dtype: object

In [18]:
import os

empresas = ["Apple", "Amazon", "Microsoft", "Alphabet", "Meta"]
df_final = df[df["empresa"].str.contains("|".join(empresas), case=False)].copy()
df_final["fuente"] = "Wikipedia: List of largest technology companies by revenue"

os.makedirs("../data/raw", exist_ok=True)
df_final.to_csv("../data/raw/wikipedia_ingresos.csv", index=False)
df_final

,empresa,ingresos_mmusd,beneficio_mmusd,pais,sede,fuente
0,Apple,416.2,112.0,US,"Cupertino, California, US",Wikipedia: List of largest technology companie...
1,Alphabet,402.8,132.2,US,"Mountain View, California, US",Wikipedia: List of largest technology companie...
2,Microsoft,281.7,101.8,US,"Redmond, Washington, US",Wikipedia: List of largest technology companie...
6,Meta,201.0,60.5,US,"Menlo Park, California, US",Wikipedia: List of largest technology companie...


In [19]:
df_final["lista_wikipedia"] = 2026
df_final.to_csv("../data/raw/wikipedia_ingresos.csv", index=False)